# Diagnostic Checks for combined_data_2018-19.csv

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path("combined_data_2018-19.csv").resolve().parent

df = pd.read_csv(DATA_DIR / "combined_data_2018-19.csv")
print(f"Shape: {df.shape}")

# --- Fix rotated columns ---
# In the raw data the 4 financial columns are rotated one position:
# payments -> receipt_cancellation -> ob_rejected -> payment_cancellation -> payments
months = ["april", "may", "june", "july", "august", "september",
          "october", "november", "december", "january", "february", "march"]
prefixes = months + ["total"]

for p in prefixes:
    pay = df[f"{p}_payments"].copy()
    df[f"{p}_payments"] = df[f"{p}_payment_cancellation"]
    df[f"{p}_payment_cancellation"] = df[f"{p}_ob_rejected"]
    df[f"{p}_ob_rejected"] = df[f"{p}_receipt_cancellation"]
    df[f"{p}_receipt_cancellation"] = pay

print(f"Rotated columns for {len(prefixes)} prefixes:")
print(f"  payments          <- payment_cancellation")
print(f"  receipt_cancellation <- payments")
print(f"  ob_rejected       <- receipt_cancellation")
print(f"  payment_cancellation <- ob_rejected")

# Shared constants used across checks
id_cols = ["financial_year", "district_name", "district_code", "block_name",
           "block_code", "panchayat_name", "panchayat_code"]

files = sorted(DATA_DIR.glob("*_monthly_summary_2018-2019.csv"))
print(f"Individual district files found: {len(files)}")

df.head()

Shape: (58691, 73)
Rotated columns for 13 prefixes:
  payments          <- payment_cancellation
  receipt_cancellation <- payments
  ob_rejected       <- receipt_cancellation
  payment_cancellation <- ob_rejected
Individual district files found: 75


,financial_year,district_name,district_code,block_name,block_code,panchayat_name,panchayat_code,opening_balance,april_receipts,april_payments,...,march_receipts,march_payments,march_receipt_cancellation,march_payment_cancellation,march_ob_rejected,total_receipts,total_payments,total_receipt_cancellation,total_payment_cancellation,total_ob_rejected
0,2018-2019,Agra,109,BARAULI AHIR,1389,AKBARPUR,42852,"259,689",666879.0,0,...,165500.0,0,1339489.0,0,0,2914226.0,0.0,2756825.0,0.0,0.0
1,2018-2019,Agra,109,BAH,1388,BICHOULA,42814,"1,513,145",378227.0,0,...,0.0,0,213898.0,0,0,1611359.0,0.0,1515742.0,0.0,0.0
2,2018-2019,Agra,109,BARAULI AHIR,1389,BARAULIAHIR,42857,"819,895.15",1178738.0,0,...,440996.0,0,2311739.0,0,0,4888356.0,0.0,4842083.0,0.0,0.0
3,2018-2019,Agra,109,BARAULI AHIR,1389,BAMRAULIKATARA,42856,"959,763.2",2048853.0,0,...,0.0,0,2433129.0,0,0,8527547.0,0.0,9259409.0,0.0,0.0
4,2018-2019,Agra,109,BARAULI AHIR,1389,BHANDAI,42860,"169,749.6",813925.0,0,...,0.0,0,0.0,0,0,2351775.0,0.0,2430644.0,0.0,0.0


## Check 1: District count — should be exactly 75

In [2]:
unique_districts = df["district_name"].nunique()
print(f"Unique districts in merged file: {unique_districts}")
if unique_districts == 75:
    print("PASS: 75 districts found")
else:
    print(f"WARNING: Expected 75 districts, got {unique_districts}")
print("\nDistrict names:")
print(sorted(df["district_name"].unique()))

Unique districts in merged file: 75
PASS: 75 districts found

District names:
['Agra', 'Aligarh', 'Ambedkar Nagar', 'Amethi', 'Amroha', 'Auraiya', 'Ayodhya', 'Azamgarh', 'Baghpat', 'Bahraich', 'Ballia', 'Balrampur', 'Banda', 'Barabanki', 'Bareilly', 'Basti', 'Bijnor', 'Budaun', 'Bulandshahr', 'Chandauli', 'Chitrakoot', 'Deoria', 'Etah', 'Etawah', 'Farrukhabad', 'Fatehpur', 'Firozabad', 'Gautam Buddha Nagar', 'Ghaziabad', 'Ghazipur', 'Gonda', 'Gorakhpur', 'Hamirpur', 'Hapur', 'Hardoi', 'Hathras', 'Jalaun', 'Jaunpur', 'Jhansi', 'Kannauj', 'Kanpur Dehat', 'Kanpur Nagar', 'Kasganj', 'Kaushambi', 'Kheri', 'Kushi Nagar', 'Lalitpur', 'Lucknow', 'Maharajganj', 'Mahoba', 'Mainpuri', 'Mathura', 'Mau', 'Meerut', 'Mirzapur', 'Moradabad', 'Muzaffarnagar', 'Pilibhit', 'Pratapgarh', 'Prayagraj', 'Rae Bareli', 'Rampur', 'Saharanpur', 'Sambhal', 'Sant Kabeer Nagar', 'Sant Ravidas Nagar', 'Shahjahanpur', 'Shamli', 'Shravasti', 'Siddharth Nagar', 'Sitapur', 'Sonbhadra', 'Sultanpur', 'Unnao', 'Varanasi']


## Check 2: District name matching — merged names vs source filenames

In [3]:
# Extract district names from filenames
filename_districts = set()
for f in files:
    name = f.name.replace("_monthly_summary_2018-2019.csv", "")
    filename_districts.add(name)

merged_districts = set(df["district_name"].unique())

in_files_not_merged = filename_districts - merged_districts
in_merged_not_files = merged_districts - filename_districts

print(f"Districts in filenames but NOT in merged data: {in_files_not_merged or 'None'}")
print(f"Districts in merged data but NOT in filenames: {in_merged_not_files or 'None'}")

if not in_files_not_merged and not in_merged_not_files:
    print("PASS: All district names match")
else:
    print("FAIL: Mismatch detected")

Districts in filenames but NOT in merged data: None
Districts in merged data but NOT in filenames: None
PASS: All district names match


## Check 3: Each file's rows only contain that district (no cross-contamination)

In [4]:
# Verify each district in merged file has rows only belonging to that district
# by checking district_name consistency per district_code
cross_check = df.groupby("district_code")["district_name"].nunique()
bad = cross_check[cross_check > 1]
if len(bad) == 0:
    print("PASS: Each district_code maps to exactly one district_name")
else:
    print("FAIL: Some district_codes map to multiple district_names:")
    for code in bad.index:
        names = df[df["district_code"] == code]["district_name"].unique()
        print(f"  district_code {code}: {names}")

PASS: Each district_code maps to exactly one district_name


## Check 4: Duplicate panchayats

## Check 4b: Panchayat code uniqueness across the whole file

In [5]:
# Check if panchayat_code is globally unique (appears in only one district+block)
code_district_counts = df.groupby("panchayat_code")["district_name"].nunique()
reused = code_district_counts[code_district_counts > 1]

if len(reused) == 0:
    print("PASS: All panchayat_codes are unique across the file")
else:
    print(f"INFO: {len(reused)} panchayat_code(s) appear in more than one district")
    print(f"(This is expected if codes are assigned within-district, not globally)\n")
    # Show which districts share each reused code
    reused_detail = (
        df[df["panchayat_code"].isin(reused.index)]
        .groupby("panchayat_code")["district_name"]
        .apply(lambda x: sorted(x.unique()))
        .reset_index()
    )
    reused_detail.columns = ["panchayat_code", "districts"]
    reused_detail["n_districts"] = reused_detail["districts"].apply(len)
    print(reused_detail.sort_values("n_districts", ascending=False).to_string(index=False))

PASS: All panchayat_codes are unique across the file


In [6]:
key_cols = ["district_code", "block_code", "panchayat_code"]
dupes = df.duplicated(subset=key_cols, keep=False)
n_dupes = dupes.sum()
if n_dupes == 0:
    print("PASS: No duplicate (district_code, block_code, panchayat_code) combinations")
else:
    print(f"FAIL: {n_dupes} duplicate rows found\n")

    # Duplicate row count per district
    dupe_by_district = (
        df[dupes]
        .groupby("district_name")
        .size()
        .reset_index(name="duplicate_rows")
        .sort_values("duplicate_rows", ascending=False)
    )
    print("Duplicate rows per district:")
    print(dupe_by_district.to_string(index=False))

    # Check if all duplicates are from Rampur / Balrampur
    dupe_districts = set(dupe_by_district["district_name"].str.lower())
    target = {"rampur", "balrampur"}
    if dupe_districts.issubset(target):
        print(f"\nAll duplicate rows are from: {sorted(dupe_districts)} — likely known data issue")
    else:
        other = sorted(dupe_districts - target)
        print(f"\nDuplicates span districts beyond Rampur/Balrampur: {other}")

FAIL: 62 duplicate rows found

Duplicate rows per district:
district_name  duplicate_rows
        Gonda              18
  Farrukhabad              16
       Etawah              10
    Firozabad               8
       Mahoba               4
       Meerut               4
      Aligarh               2

Duplicates span districts beyond Rampur/Balrampur: ['aligarh', 'etawah', 'farrukhabad', 'firozabad', 'gonda', 'mahoba', 'meerut']


## Check 4c: Drop exact duplicate rows & investigate remaining panchayat-code duplicates

In [7]:
# --- Step A: Drop full (exact) duplicate rows ---
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
exact_dupes_dropped = before - after
print(f"Dropped {exact_dupes_dropped} exact duplicate rows ({before} → {after})")

# --- Step B: Check for remaining panchayat-code duplicates (same key, different data) ---
key_cols = ["district_code", "block_code", "panchayat_code"]
remaining_dupes = df.duplicated(subset=key_cols, keep=False)
n_remaining = remaining_dupes.sum()

if n_remaining == 0:
    print(f"\nPASS: No remaining panchayat-code duplicates after dedup")
else:
    print(f"\nWARNING: {n_remaining} rows still share a (district_code, block_code, panchayat_code) key")
    print("These have the SAME key but DIFFERENT data — not exact duplicates.\n")

    dup_df = df[remaining_dupes].sort_values(key_cols)
    dup_groups = dup_df.groupby(key_cols)

    print(f"Number of duplicate groups: {len(dup_groups)}")
    print(f"Districts affected:")
    print(dup_df.groupby("district_name").size().sort_values(ascending=False).to_string())

    # Show a few example groups with the columns that differ
    fin_cols = ["opening_balance", "total_receipts", "total_payments",
                "total_receipt_cancellation", "total_payment_cancellation"]
    compare_cols = ["district_name", "block_name", "panchayat_name", "panchayat_code"] + fin_cols

    print(f"\nSample near-duplicate groups (first 5):")
    for idx, (name, group) in enumerate(dup_groups):
        if idx >= 5:
            break
        print(f"\n  Group {idx+1} — key {name}:")
        print(group[compare_cols].to_string(index=False))

        # Quantify how different the financial values are
        for col in fin_cols:
            vals = group[col].dropna().unique()
            if len(vals) > 1:
                print(f"    → '{col}' differs: {vals}")

print(f"\nShape after dedup: {df.shape}")

Dropped 27 exact duplicate rows (58691 → 58664)

These have the SAME key but DIFFERENT data — not exact duplicates.

Number of duplicate groups: 4
Districts affected:
district_name
Etawah     6
Aligarh    2

Sample near-duplicate groups (first 5):

  Group 1 — key (110, 1404, 43591):
district_name block_name panchayat_name  panchayat_code opening_balance  total_receipts  total_payments  total_receipt_cancellation  total_payment_cancellation
      Aligarh   CHANDAUS       MOHARANA           43591         417,624       1734357.0             0.0                   1954662.5                         0.0
      Aligarh   CHANDAUS       MOHRAINA           43591         417,624       1734357.0             0.0                   1954662.5                         0.0

  Group 2 — key (130, 1661, 60522):
district_name block_name  panchayat_name  panchayat_code opening_balance  total_receipts  total_payments  total_receipt_cancellation  total_payment_cancellation
       Etawah   BASREHAR SHERPURBASGA

## Check 5: Missing values in key identifier columns

In [8]:
# Check for and remove "district total" summary rows before checking missing values
for col in ["district_name", "panchayat_name", "block_name"]:
    if col in df.columns:
        mask = df[col].astype(str).str.contains("district total", case=False, na=False)
        n_found = mask.sum()
        if n_found > 0:
            print(f"Found {n_found} 'district total' rows in '{col}' — removing them")
            print(df[mask][["district_name", "block_name", "panchayat_name"]].to_string())
            df = df[~mask].reset_index(drop=True)
        else:
            print(f"No 'district total' rows found in '{col}'")

print(f"\nShape after removal: {df.shape}")

# Now check for missing values in key identifier columns
missing = df[id_cols].isnull().sum()
if missing.sum() == 0:
    print("\nPASS: No missing values in identifier columns")
else:
    print("\nFAIL: Missing values found:")
    print(missing[missing > 0])

No 'district total' rows found in 'district_name'
No 'district total' rows found in 'panchayat_name'
No 'district total' rows found in 'block_name'

Shape after removal: (58664, 73)

PASS: No missing values in identifier columns


## Check 6: Financial year consistency

In [9]:
fy_values = df["financial_year"].unique()
print(f"Unique financial_year values: {fy_values}")
if len(fy_values) == 1 and fy_values[0] == "2018-2019":
    print("PASS: All rows are 2018-2019")
else:
    print("FAIL: Unexpected financial_year values found")
    print(df["financial_year"].value_counts())

Unique financial_year values: ['2018-2019']
PASS: All rows are 2018-2019


## Check 7: opening_balance parsing (Indian comma format)

In [10]:
# opening_balance is likely read as string due to Indian comma formatting ("29,23,405.00")
print(f"opening_balance dtype: {df['opening_balance'].dtype}")
print(f"Sample values:\n{df['opening_balance'].head(10)}")

# Check if it's string type and needs cleaning
if df["opening_balance"].dtype == object:
    print("\nWARNING: opening_balance is a string column (Indian comma format)")
    ob_cleaned = pd.to_numeric(
        df["opening_balance"].str.replace(",", "", regex=False), errors="coerce"
    )
    df["opening_balance"] = ob_cleaned
    print(f"After cleaning — min: {ob_cleaned.min()}, max: {ob_cleaned.max()}, nulls: {ob_cleaned.isnull().sum()}")
    n_negative = (ob_cleaned < 0).sum()
    print(f"Negative opening balances: {n_negative}")
else:
    print("opening_balance is already numeric")
    print(f"min: {df['opening_balance'].min()}, max: {df['opening_balance'].max()}")

# Always create an explicit float column for downstream use
df["opening_balance_numeric"] = pd.to_numeric(df["opening_balance"], errors="coerce")
print(f"\nopening_balance_numeric created — dtype: {df['opening_balance_numeric'].dtype}, nulls: {df['opening_balance_numeric'].isnull().sum()}")

opening_balance dtype: object
Sample values:
0         259,689
1       1,513,145
2      819,895.15
3       959,763.2
4       169,749.6
5      421,555.34
6      285,630.14
7       59,915.29
8    6,631,842.79
9    2,363,139.62
Name: opening_balance, dtype: object

After cleaning — min: 0.0, max: 977542093.7, nulls: 25
Negative opening balances: 0

opening_balance_numeric created — dtype: float64, nulls: 25


## Check 8: Monthly totals consistency (row-level arithmetic check)

In [11]:
months = ["april", "may", "june", "july", "august", "september",
          "october", "november", "december", "january", "february", "march"]

check_fields = ["receipts", "payments", "receipt_cancellation", "payment_cancellation", "ob_rejected"]

all_pass = True
for field in check_fields:
    monthly_cols = [f"{m}_{field}" for m in months]
    total_col = f"total_{field}"
    
    computed_total = df[monthly_cols].sum(axis=1)
    reported_total = df[total_col]
    
    # Allow small floating point tolerance
    mismatches = ~np.isclose(computed_total, reported_total, atol=0.01, equal_nan=True)
    n_bad = mismatches.sum()
    
    if n_bad == 0:
        print(f"PASS: {total_col} matches sum of monthly columns")
    else:
        all_pass = False
        print(f"FAIL: {total_col} — {n_bad} rows have mismatched totals")
        bad_rows = df[mismatches][["district_name", "panchayat_name", total_col]].copy()
        bad_rows["computed"] = computed_total[mismatches]
        bad_rows["diff"] = bad_rows["computed"] - bad_rows[total_col]
        print(bad_rows.head(10))
        print()

if all_pass:
    print("\nAll monthly-to-total checks passed!")

PASS: total_receipts matches sum of monthly columns
PASS: total_payments matches sum of monthly columns
PASS: total_receipt_cancellation matches sum of monthly columns
PASS: total_payment_cancellation matches sum of monthly columns
PASS: total_ob_rejected matches sum of monthly columns

All monthly-to-total checks passed!


## Check 9: Negative values in financial columns

In [12]:
# All monthly receipt/payment columns plus totals should be >= 0
# Use select_dtypes to avoid errors on any remaining object columns
numeric_cols = df.select_dtypes(include="number").columns.difference(["opening_balance"])
neg_report = {}
for col in numeric_cols:
    n_neg = (df[col] < 0).sum()
    if n_neg > 0:
        neg_report[col] = n_neg

if not neg_report:
    print("PASS: No negative values in any financial column")
else:
    print(f"FAIL: Negative values found in {len(neg_report)} columns:")
    for col, count in neg_report.items():
        print(f"  {col}: {count} negative values")

PASS: No negative values in any financial column


## Check 10: Panchayat count per district — compare merged vs individual files

In [13]:
# Count rows per district in merged file
merged_counts = df.groupby("district_name").size().to_dict()

# Count rows per district from individual files
# engine="python" is more robust for malformed/non-standard CSVs
file_counts = {}
skipped = {}
for f in files:
    name = f.name.replace("_monthly_summary_2018-2019.csv", "")
    individual = pd.read_csv(f, encoding="latin-1", engine="python", on_bad_lines="skip")
    file_counts[name] = len(individual)
    # Warn if any lines were skipped due to parse errors
    with open(f, encoding="latin-1") as fh:
        raw_rows = sum(1 for _ in fh) - 1  # subtract header
    if raw_rows != len(individual):
        skipped[name] = raw_rows - len(individual)

if skipped:
    print("WARNING: Bad lines skipped in the following files:")
    for name, n in skipped.items():
        print(f"  {name}: {n} line(s) skipped")
    print()

# Compare
mismatches = []
for district in sorted(set(list(merged_counts.keys()) + list(file_counts.keys()))):
    mc = merged_counts.get(district, 0)
    fc = file_counts.get(district, 0)
    if mc != fc:
        mismatches.append((district, fc, mc))

if not mismatches:
    print(f"PASS: Row counts match for all {len(merged_counts)} districts")
else:
    print(f"FAIL: Row count mismatches in {len(mismatches)} districts:")
    mismatch_df = pd.DataFrame(mismatches, columns=["district", "individual_file_rows", "merged_rows"])
    print(mismatch_df.to_string(index=False))

FAIL: Row count mismatches in 6 districts:
   district  individual_file_rows  merged_rows
     Etawah                   476          474
Farrukhabad                   608          600
  Firozabad                   572          568
      Gonda                  1062         1053
     Mahoba                   275          273
     Meerut                   481          479


## Check 11: Summary statistics for quick sanity check

## Clean & Compute: Apply script fixes to merged df

In [14]:
# --- Step 1: District total rows ---
# Already removed in Check 5, confirm none remain
remaining = df["panchayat_name"].astype(str).str.contains("district total", case=False, na=False).sum()
print(f"District total rows remaining: {remaining} (should be 0)")

# --- Step 2: Clean all 5 financial columns ---
fin_cols = [
    "total_payments", "total_payment_cancellation",
    "opening_balance", "total_receipts", "total_receipt_cancellation"
]

for col in fin_cols:
    if col not in df.columns:
        print(f"ERROR: missing column '{col}'")
        raise KeyError(col)
    if df[col].dtype == object:
        print(f"Cleaning '{col}' (Indian comma format)...")
        df[col] = pd.to_numeric(df[col].str.replace(",", "", regex=False), errors="coerce")
    else:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    n_nan = df[col].isna().sum()
    if n_nan > 0:
        print(f"  Warning: {n_nan} missing values in '{col}' kept as NaN")

print("\nAll financial columns cleaned.")

# --- Step 3: Compute utilization_index (with cancellations) ---
ui_num = df["total_payments"] - df["total_payment_cancellation"]
ui_den = df["opening_balance"] + df["total_receipts"] - df["total_receipt_cancellation"]
df["utilization_index"] = np.where(ui_den <= 0, np.nan, ui_num / ui_den)

print(f"\nutilization_index (with cancellations):")
print(f"  Zero/neg denominators -> NaN: {(ui_den <= 0).sum()}")
print(f"  Valid rows: {df['utilization_index'].notna().sum()} / {len(df)}")
print(f"  Min: {df['utilization_index'].min():.4f}, Median: {df['utilization_index'].median():.4f}, "
      f"Max: {df['utilization_index'].max():.4f}, Mean: {df['utilization_index'].mean():.4f}")

# --- Step 4: Compute utilization_index_no_canc (without cancellations) ---
ui_nc_num = df["total_payments"]
ui_nc_den = df["opening_balance"] + df["total_receipts"]
df["utilization_index_no_canc"] = np.where(ui_nc_den <= 0, np.nan, ui_nc_num / ui_nc_den)

print(f"\nutilization_index_no_canc (without cancellations):")
print(f"  Zero/neg denominators -> NaN: {(ui_nc_den <= 0).sum()}")
print(f"  Valid rows: {df['utilization_index_no_canc'].notna().sum()} / {len(df)}")
print(f"  Min: {df['utilization_index_no_canc'].min():.4f}, Median: {df['utilization_index_no_canc'].median():.4f}, "
      f"Max: {df['utilization_index_no_canc'].max():.4f}, Mean: {df['utilization_index_no_canc'].mean():.4f}")

# --- Step 5: Compute payment_ratio (with cancellations) ---
pr_num = df["total_payments"] - df["total_payment_cancellation"]
pr_den = df["total_receipts"] - df["total_receipt_cancellation"]
df["payment_ratio"] = np.where(pr_den <= 0, np.nan, pr_num / pr_den)

print(f"\npayment_ratio (with cancellations):")
print(f"  Zero/neg denominators -> NaN: {(pr_den <= 0).sum()}")
print(f"  Valid rows: {df['payment_ratio'].notna().sum()} / {len(df)}")
print(f"  Min: {df['payment_ratio'].min():.4f}, Median: {df['payment_ratio'].median():.4f}, "
      f"Max: {df['payment_ratio'].max():.4f}, Mean: {df['payment_ratio'].mean():.4f}")

# --- Step 6: Compute payment_ratio_no_canc (without cancellations) ---
pr_nc_num = df["total_payments"]
pr_nc_den = df["total_receipts"]
df["payment_ratio_no_canc"] = np.where(pr_nc_den <= 0, np.nan, pr_nc_num / pr_nc_den)

print(f"\npayment_ratio_no_canc (without cancellations):")
print(f"  Zero/neg denominators -> NaN: {(pr_nc_den <= 0).sum()}")
print(f"  Valid rows: {df['payment_ratio_no_canc'].notna().sum()} / {len(df)}")
print(f"  Min: {df['payment_ratio_no_canc'].min():.4f}, Median: {df['payment_ratio_no_canc'].median():.4f}, "
      f"Max: {df['payment_ratio_no_canc'].max():.4f}, Mean: {df['payment_ratio_no_canc'].mean():.4f}")

# --- Summary table with out-of-range checks ---
metrics = ["utilization_index", "utilization_index_no_canc", "payment_ratio", "payment_ratio_no_canc"]
print(f"\n{'Metric':<30s} {'Valid':>6s} {'NaN':>6s} {'<0':>6s} {'>1':>6s} {'Min':>10s} {'Median':>10s} {'Max':>10s}")
print("-" * 90)
for m in metrics:
    col = df[m]
    v   = col.notna().sum()
    n   = col.isna().sum()
    neg = (col < 0).sum()
    gt1 = (col > 1).sum()
    flag = " <-- OUT OF RANGE" if (neg > 0 or gt1 > 0) else ""
    print(f"{m:<30s} {v:>6d} {n:>6d} {neg:>6d} {gt1:>6d} {col.min():>10.4f} {col.median():>10.4f} {col.max():>10.4f}{flag}")

# Show rows where any metric is out of [0, 1]
out_of_range_mask = (
    (df[metrics] < 0).any(axis=1) | (df[metrics] > 1).any(axis=1)
)
n_oor = out_of_range_mask.sum()
print(f"\nRows with any metric outside [0, 1]: {n_oor}")
if n_oor > 0:
    print(df[out_of_range_mask][["district_name", "panchayat_name"] + metrics].head(20).to_string(index=False))

df[["district_name", "panchayat_name"] + metrics].head(10)

District total rows remaining: 0 (should be 0)

All financial columns cleaned.

utilization_index (with cancellations):
  Zero/neg denominators -> NaN: 35
  Valid rows: 58604 / 58664
  Min: 0.0000, Median: 0.0000, Max: 0.0000, Mean: 0.0000

utilization_index_no_canc (without cancellations):
  Zero/neg denominators -> NaN: 0
  Valid rows: 58639 / 58664
  Min: 0.0000, Median: 0.0000, Max: 0.0000, Mean: 0.0000

payment_ratio (with cancellations):
  Zero/neg denominators -> NaN: 32627
  Valid rows: 26037 / 58664
  Min: 0.0000, Median: 0.0000, Max: 0.0000, Mean: 0.0000

payment_ratio_no_canc (without cancellations):
  Zero/neg denominators -> NaN: 32
  Valid rows: 58632 / 58664
  Min: 0.0000, Median: 0.0000, Max: 0.0000, Mean: 0.0000

Metric                          Valid    NaN     <0     >1        Min     Median        Max
------------------------------------------------------------------------------------------
utilization_index               58604     60      0      0     0.0000     0.0

,district_name,panchayat_name,utilization_index,utilization_index_no_canc,payment_ratio,payment_ratio_no_canc
0,Agra,AKBARPUR,0.0,0.0,0.0,0.0
1,Agra,BICHOULA,0.0,0.0,0.0,0.0
2,Agra,BARAULIAHIR,0.0,0.0,0.0,0.0
3,Agra,BAMRAULIKATARA,0.0,0.0,NaN,0.0
4,Agra,BHANDAI,0.0,0.0,NaN,0.0
5,Agra,AKHBAI,0.0,0.0,NaN,0.0
6,Agra,Bhartar,0.0,0.0,0.0,0.0
7,Agra,BASAIBOBLA,0.0,0.0,0.0,0.0
8,Agra,BHADROLI,0.0,0.0,NaN,0.0
9,Agra,ATUS,0.0,0.0,NaN,0.0


## Dummy variable: utilization_index > 1

In [15]:
# Dummy = 1 if utilization_index > 1, 0 otherwise, NaN if UI is NaN
df["ui_over1"] = np.where(df["utilization_index"].isna(), np.nan,
                          (df["utilization_index"] > 1).astype(int))

n_over1 = (df["ui_over1"] == 1).sum()
n_valid = df["ui_over1"].notna().sum()
print(f"ui_over1 dummy created:")
print(f"  1 (UI > 1):  {n_over1} ({n_over1/n_valid*100:.1f}%)")
print(f"  0 (UI <= 1): {(df['ui_over1'] == 0).sum()} ({(df['ui_over1'] == 0).sum()/n_valid*100:.1f}%)")
print(f"  NaN:         {df['ui_over1'].isna().sum()}")

# Breakdown by district
dist_over1 = (
    df.groupby("district_name")
    .agg(
        n_panchayats=("ui_over1", "count"),
        n_over1=("ui_over1", lambda x: (x == 1).sum()),
    )
)
dist_over1["pct_over1"] = (dist_over1["n_over1"] / dist_over1["n_panchayats"] * 100).round(1)
dist_over1 = dist_over1.sort_values("pct_over1", ascending=False)

print(f"\nTop 10 districts by % of panchayats with UI > 1:")
print(dist_over1.head(10).to_string())
print(f"\nBottom 5 districts:")
print(dist_over1.tail(5).to_string())

ui_over1 dummy created:
  1 (UI > 1):  0 (0.0%)
  0 (UI <= 1): 58604 (100.0%)
  NaN:         60

Top 10 districts by % of panchayats with UI > 1:
               n_panchayats  n_over1  pct_over1
district_name                                  
Agra                    695        0        0.0
Muzaffarnagar           498        0        0.0
Mirzapur                809        0        0.0
Meerut                  479        0        0.0
Mau                     675        0        0.0
Mathura                 516        0        0.0
Mainpuri                549        0        0.0
Mahoba                  273        0        0.0
Maharajganj             923        0        0.0
Lucknow                 570        0        0.0

Bottom 5 districts:
               n_panchayats  n_over1  pct_over1
district_name                                  
Farrukhabad             600        0        0.0
Etawah                  450        0        0.0
Etah                    574        0        0.0
Deoria           

In [16]:
# Overall summary
print(f"Total rows: {len(df)}")
print(f"Total districts: {df['district_name'].nunique()}")
print(f"Total blocks: {df['block_name'].nunique()}")
# Use composite key for panchayat uniqueness — names like "Ward 1" repeat across districts
n_unique_panchayats = df[["district_code", "block_code", "panchayat_code"]].drop_duplicates().shape[0]
print(f"Total unique panchayats (by composite key): {n_unique_panchayats}")
print(f"\nRows per district (min/max/mean):")
dist_sizes = df.groupby("district_name").size()
print(f"  Min: {dist_sizes.min()} ({dist_sizes.idxmin()})")
print(f"  Max: {dist_sizes.max()} ({dist_sizes.idxmax()})")
print(f"  Mean: {dist_sizes.mean():.1f}")
print(f"\nTotal NaN values across entire dataframe: {df.isnull().sum().sum()}")
print(f"\nColumn dtypes:\n{df.dtypes}")

Total rows: 58664
Total districts: 75
Total blocks: 796
Total unique panchayats (by composite key): 58660

Rows per district (min/max/mean):
  Min: 100 (Gautam Buddha Nagar)
  Max: 1869 (Azamgarh)
  Mean: 782.2

Total NaN values across entire dataframe: 32854

Column dtypes:
financial_year                object
district_name                 object
district_code                  int64
block_name                    object
block_code                     int64
                              ...   
utilization_index            float64
utilization_index_no_canc    float64
payment_ratio                float64
payment_ratio_no_canc        float64
ui_over1                     float64
Length: 79, dtype: object


---
# Round 2: Structural & Distribution Checks (post-cleaning)

## Check 12: District-level aggregate outliers

In [17]:
# Flag districts whose aggregate total_payments or total_receipts
# are >3 std deviations from the cross-district mean
# (could indicate leftover DISTRICT TOTAL rows or genuinely anomalous districts)

dist_agg = df.groupby("district_name").agg(
    total_payments_sum=("total_payments", "sum"),
    total_receipts_sum=("total_receipts", "sum"),
    n_panchayats=("panchayat_name", "size"),
).reset_index()

for col in ["total_payments_sum", "total_receipts_sum"]:
    mean = dist_agg[col].mean()
    std = dist_agg[col].std()
    outliers = dist_agg[np.abs(dist_agg[col] - mean) > 3 * std]
    if len(outliers) == 0:
        print(f"PASS: No district-level outliers for {col} (>3σ)")
    else:
        print(f"WARNING: {len(outliers)} district(s) are outliers for {col} (>3σ from mean):")
        print(f"  Mean: {mean:,.0f}, Std: {std:,.0f}")
        print(outliers[["district_name", col, "n_panchayats"]].to_string(index=False))
    print()

PASS: No district-level outliers for total_payments_sum (>3σ)

PASS: No district-level outliers for total_receipts_sum (>3σ)



## Check 12b: Diagnose district-level outliers (Ghazipur payments, Bahraich receipts)

In [18]:
# --- Diagnose Ghazipur (total_payments outlier) ---
print("=" * 70)
print("DIAGNOSING: Ghazipur — total_payments outlier")
print("=" * 70)

ghz = df[df["district_name"].str.contains("Ghazipur", case=False, na=False)]
print(f"\nRows in Ghazipur: {len(ghz)}")
print(f"Total payments sum: {ghz['total_payments'].sum():,.0f}")

# Distribution of panchayat-level total_payments
print(f"\nPanchayat-level total_payments distribution:")
print(ghz["total_payments"].describe().to_string())

# Top panchayats driving the sum
top_ghz = ghz.nlargest(10, "total_payments")[["block_name", "panchayat_name", "panchayat_code", "total_payments"]]
print(f"\nTop 10 panchayats by total_payments:")
print(top_ghz.to_string(index=False))

# Check for extreme individual panchayats (>3σ within district)
ghz_mean = ghz["total_payments"].mean()
ghz_std = ghz["total_payments"].std()
ghz_outliers = ghz[ghz["total_payments"] > ghz_mean + 3 * ghz_std]
print(f"\nPanchayats >3σ within Ghazipur (mean={ghz_mean:,.0f}, std={ghz_std:,.0f}): {len(ghz_outliers)}")
if len(ghz_outliers) > 0:
    print(ghz_outliers[["block_name", "panchayat_name", "panchayat_code", "total_payments"]].to_string(index=False))

# Compare per-panchayat average to other districts
print(f"\nPer-panchayat average comparison:")
per_panch = dist_agg.copy()
per_panch["payments_per_panchayat"] = per_panch["total_payments_sum"] / per_panch["n_panchayats"]
per_panch_sorted = per_panch.sort_values("payments_per_panchayat", ascending=False).head(10)
print(per_panch_sorted[["district_name", "payments_per_panchayat", "n_panchayats", "total_payments_sum"]].to_string(index=False))

print("\n")
# --- Diagnose Bahraich (total_receipts outlier) ---
print("=" * 70)
print("DIAGNOSING: Bahraich — total_receipts outlier")
print("=" * 70)

bhr = df[df["district_name"].str.contains("Bahraich", case=False, na=False)]
print(f"\nRows in Bahraich: {len(bhr)}")
print(f"Total receipts sum: {bhr['total_receipts'].sum():,.0f}")

# Distribution of panchayat-level total_receipts
print(f"\nPanchayat-level total_receipts distribution:")
print(bhr["total_receipts"].describe().to_string())

# Top panchayats driving the sum
top_bhr = bhr.nlargest(10, "total_receipts")[["block_name", "panchayat_name", "panchayat_code", "total_receipts"]]
print(f"\nTop 10 panchayats by total_receipts:")
print(top_bhr.to_string(index=False))

# Check for extreme individual panchayats (>3σ within district)
bhr_mean = bhr["total_receipts"].mean()
bhr_std = bhr["total_receipts"].std()
bhr_outliers = bhr[bhr["total_receipts"] > bhr_mean + 3 * bhr_std]
print(f"\nPanchayats >3σ within Bahraich (mean={bhr_mean:,.0f}, std={bhr_std:,.0f}): {len(bhr_outliers)}")
if len(bhr_outliers) > 0:
    print(bhr_outliers[["block_name", "panchayat_name", "panchayat_code", "total_receipts"]].to_string(index=False))

# Compare per-panchayat average to other districts
print(f"\nPer-panchayat average comparison:")
per_panch["receipts_per_panchayat"] = per_panch["total_receipts_sum"] / per_panch["n_panchayats"]
per_panch_sorted = per_panch.sort_values("receipts_per_panchayat", ascending=False).head(10)
print(per_panch_sorted[["district_name", "receipts_per_panchayat", "n_panchayats", "total_receipts_sum"]].to_string(index=False))

# Check monthly pattern for Bahraich — is the spike in one month?
print(f"\nBahraich monthly receipt totals (summed across all panchayats):")
months = ["april", "may", "june", "july", "august", "september",
          "october", "november", "december", "january", "february", "march"]
receipt_cols = [f"{m}_receipts" for m in months if f"{m}_receipts" in df.columns]
if receipt_cols:
    monthly_totals = bhr[receipt_cols].sum()
    print(monthly_totals.to_string())
    print(f"\nMonth with max receipts: {monthly_totals.idxmax()} = {monthly_totals.max():,.0f}")

DIAGNOSING: Ghazipur — total_payments outlier

Rows in Ghazipur: 1235
Total payments sum: 0

Panchayat-level total_payments distribution:
count    1235.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0

Top 10 panchayats by total_payments:
block_name panchayat_name  panchayat_code  total_payments
BHANWARKOL        AALAPUR           63870             0.0
  BHADAURA      BHADAURAA           63838             0.0
  BHADAURA   BHABHANAULIA           63837             0.0
  JAKHANIA         BAGHAI           64069             0.0
  GHAZIPUR     BANZARIPUR           64010             0.0
  BHADAURA         AMAURA           63830             0.0
BHANWARKOL  AWATHHIBASANT           63872             0.0
   DEVKALI       AALAMPUR           63928             0.0
  BHADAURA         ARANGI           63831             0.0
  BHADAURA       BAHUWARA           63832             0.0

Panchayats >3σ within Ghazipur (mean=0, std=0): 0

Per-pa

## Check 13: Zero metric values — breakdown by district (all 4 metrics)

In [19]:
# Panchayats where metric == 0 exactly: had available funds but zero net payments
metrics = ["utilization_index", "utilization_index_no_canc", "payment_ratio", "payment_ratio_no_canc"]
dist_total = df.groupby("district_name").size().reset_index(name="total_panchayats")

for metric in metrics:
    zero_m = df[df[metric] == 0]
    print(f"{'='*70}")
    print(f"{metric}: {len(zero_m)} panchayats with value == 0 / {len(df)}")
    
    if len(zero_m) > 0:
        zero_by_dist = (
            zero_m.groupby("district_name").size()
            .reset_index(name="zero_count")
            .sort_values("zero_count", ascending=False)
        )
        zero_by_dist = zero_by_dist.merge(dist_total, on="district_name")
        zero_by_dist["pct_zero"] = (zero_by_dist["zero_count"] / zero_by_dist["total_panchayats"] * 100).round(1)
        
        print(f"\nTop 10 districts with most zero-value panchayats:")
        print(zero_by_dist.head(10).to_string(index=False))
        
        high_zero = zero_by_dist[zero_by_dist["pct_zero"] > 20]
        if len(high_zero) > 0:
            print(f"\nWARNING: {len(high_zero)} district(s) have >20% zero-value panchayats")
    else:
        print("PASS: No panchayats with exactly zero value")
    print()

utilization_index: 58604 panchayats with value == 0 / 58664

Top 10 districts with most zero-value panchayats:
  district_name  zero_count  total_panchayats  pct_zero
       Azamgarh        1868              1869      99.9
        Jaunpur        1746              1746     100.0
      Prayagraj        1636              1636     100.0
        Sitapur        1581              1581     100.0
      Gorakhpur        1325              1343      98.7
         Hardoi        1305              1305     100.0
     Pratapgarh        1241              1241     100.0
       Ghazipur        1235              1235     100.0
Siddharth Nagar        1196              1199      99.7
       Bareilly        1193              1193     100.0


utilization_index_no_canc: 58639 panchayats with value == 0 / 58664

Top 10 districts with most zero-value panchayats:
  district_name  zero_count  total_panchayats  pct_zero
       Azamgarh        1869              1869     100.0
        Jaunpur        1746             

## Check 14: NaN values — diagnose why (all 4 metrics)

In [20]:
# NaN comes from denominator <= 0. Break down the reasons for each metric.
# Denominators:
#   utilization_index:          OB + total_receipts - total_receipt_cancellation
#   utilization_index_no_canc:  OB + total_receipts
#   payment_ratio:              total_receipts - total_receipt_cancellation
#   payment_ratio_no_canc:      total_receipts

metric_denom = {
    "utilization_index":          ("opening_balance", "total_receipts", "total_receipt_cancellation", True),
    "utilization_index_no_canc":  ("opening_balance", "total_receipts", None, False),
    "payment_ratio":              (None, "total_receipts", "total_receipt_cancellation", True),
    "payment_ratio_no_canc":      (None, "total_receipts", None, False),
}

for metric, (ob_col, rec_col, canc_col, uses_canc) in metric_denom.items():
    nan_m = df[df[metric].isna()].copy()
    print(f"{'='*70}")
    print(f"{metric}: {len(nan_m)} NaN rows / {len(df)}\n")
    
    if len(nan_m) == 0:
        print("PASS: No NaN values\n")
        continue
    
    # Compute denominator for this metric
    denom = pd.Series(0.0, index=nan_m.index)
    if ob_col:
        denom += nan_m[ob_col]
    denom += nan_m[rec_col]
    if canc_col:
        denom -= nan_m[canc_col]
    
    # Reason breakdown
    no_funds = True
    if ob_col:
        no_funds = (nan_m[ob_col] == 0) & (nan_m[rec_col] == 0)
    else:
        no_funds = nan_m[rec_col] == 0
    print(f"  No funds at all (zero inputs): {no_funds.sum()}")
    
    if uses_canc and canc_col:
        canc_wiped = (~no_funds) & (denom <= 0)
        print(f"  Had funds but cancellations wiped them out: {canc_wiped.sum()}")
    
    neg_d = (denom < 0).sum()
    print(f"  Negative denominator: {neg_d}")
    
    # NaN in underlying columns
    check_cols = [c for c in [ob_col, rec_col, canc_col] if c]
    has_nan_cols = nan_m[check_cols].isna().any(axis=1).sum()
    print(f"  NaN in underlying financial columns: {has_nan_cols}")
    
    # District breakdown (top 10)
    print(f"\n  NaN by district (top 10):")
    nan_by_dist = nan_m.groupby("district_name").size().sort_values(ascending=False).head(10)
    for dist, count in nan_by_dist.items():
        total = len(df[df["district_name"] == dist])
        print(f"    {dist}: {count}/{total} ({count/total*100:.1f}%)")
    print()

utilization_index: 60 NaN rows / 58664

  No funds at all (zero inputs): 0
  Had funds but cancellations wiped them out: 35
  Negative denominator: 0
  NaN in underlying financial columns: 25

  NaN by district (top 10):
    Etawah: 24/474 (5.1%)
    Gorakhpur: 18/1343 (1.3%)
    Bahraich: 7/1054 (0.7%)
    Siddharth Nagar: 3/1199 (0.3%)
    Kheri: 2/1167 (0.2%)
    Ambedkar Nagar: 1/927 (0.1%)
    Amethi: 1/682 (0.1%)
    Amroha: 1/601 (0.2%)
    Azamgarh: 1/1869 (0.1%)
    Etah: 1/575 (0.2%)

utilization_index_no_canc: 25 NaN rows / 58664

  No funds at all (zero inputs): 0
  Negative denominator: 0
  NaN in underlying financial columns: 25

  NaN by district (top 10):
    Etawah: 24/474 (5.1%)
    Ambedkar Nagar: 1/927 (0.1%)

payment_ratio: 32627 NaN rows / 58664

  No funds at all (zero inputs): 32
  Had funds but cancellations wiped them out: 32595
  Negative denominator: 32294
  NaN in underlying financial columns: 0

  NaN by district (top 10):
    Azamgarh: 1292/1869 (69.1%)
 

## Check 15: Metric > 1 — breakdown by district (all 4 metrics)

In [21]:
# Metric > 1 means spending exceeded available funds (UI) or receipts (PR)
# A few is normal (timing), concentrated in one district = data issue
metrics = ["utilization_index", "utilization_index_no_canc", "payment_ratio", "payment_ratio_no_canc"]
dist_total = df.groupby("district_name").size().reset_index(name="total_panchayats")

for metric in metrics:
    over1 = df[df[metric] > 1]
    print(f"{'='*70}")
    print(f"{metric} > 1: {len(over1)} panchayats / {len(df)}")
    
    if len(over1) > 0:
        over1_by_dist = (
            over1.groupby("district_name").size()
            .reset_index(name="over1_count")
            .sort_values("over1_count", ascending=False)
        )
        over1_by_dist = over1_by_dist.merge(dist_total, on="district_name")
        over1_by_dist["pct_over1"] = (over1_by_dist["over1_count"] / over1_by_dist["total_panchayats"] * 100).round(1)
        
        print(f"\nTop 10 districts with most >1 panchayats:")
        print(over1_by_dist.head(10).to_string(index=False))
        
        print(f"\nTop 5 highest {metric} values:")
        extreme = df.nlargest(5, metric)[
            ["district_name", "block_name", "panchayat_name", metric,
             "opening_balance", "total_receipts", "total_payments"]
        ]
        print(extreme.to_string(index=False))
    else:
        print("PASS: No values > 1")
    print()

utilization_index > 1: 0 panchayats / 58664
PASS: No values > 1

utilization_index_no_canc > 1: 0 panchayats / 58664
PASS: No values > 1

payment_ratio > 1: 0 panchayats / 58664
PASS: No values > 1

payment_ratio_no_canc > 1: 0 panchayats / 58664
PASS: No values > 1



### Check 15b: Diagnose UI > 1 — inflated payments vs understated funds + district distribution

In [22]:
# =====================================================================
# PART A: Are payments inflated or is OB+receipts understated?
# =====================================================================
# UI = (payments - pay_canc) / (OB + receipts - rec_canc)
# UI > 1 means numerator > denominator
# Compare: how much of the "excess" comes from high payments vs low funds?

over1 = df[df["utilization_index"] > 1].copy()
normal = df[(df["utilization_index"] > 0) & (df["utilization_index"] <= 1)].copy()

print(f"Rows with UI > 1: {len(over1)}")
print(f"Rows with 0 < UI <= 1: {len(normal)}\n")

# Compare median financial values: over1 vs normal
compare_cols = ["opening_balance", "total_receipts", "total_payments",
                "total_payment_cancellation", "total_receipt_cancellation"]

print(f"{'Column':<30s} {'Median (UI>1)':>15s} {'Median (0<UI<=1)':>18s} {'Ratio':>8s}")
print("-" * 75)
for col in compare_cols:
    med_over = over1[col].median()
    med_norm = normal[col].median()
    ratio = med_over / med_norm if med_norm != 0 else np.nan
    print(f"{col:<30s} {med_over:>15,.0f} {med_norm:>18,.0f} {ratio:>8.2f}x")

# Net numerator and denominator
over1["net_payments"] = over1["total_payments"] - over1["total_payment_cancellation"]
over1["available_funds"] = over1["opening_balance"] + over1["total_receipts"] - over1["total_receipt_cancellation"]
over1["excess"] = over1["net_payments"] - over1["available_funds"]

normal["net_payments"] = normal["total_payments"] - normal["total_payment_cancellation"]
normal["available_funds"] = normal["opening_balance"] + normal["total_receipts"] - normal["total_receipt_cancellation"]

print(f"\n{'Derived':<30s} {'Median (UI>1)':>15s} {'Median (0<UI<=1)':>18s} {'Ratio':>8s}")
print("-" * 75)
for col in ["net_payments", "available_funds"]:
    med_over = over1[col].median()
    med_norm = normal[col].median()
    ratio = med_over / med_norm if med_norm != 0 else np.nan
    print(f"{col:<30s} {med_over:>15,.0f} {med_norm:>18,.0f} {ratio:>8.2f}x")

print(f"\nMedian excess spending (net_payments - available_funds): {over1['excess'].median():,.0f}")
print(f"Mean excess spending: {over1['excess'].mean():,.0f}")

# What fraction of over1 panchayats have OB == 0?
ob_zero_pct = (over1["opening_balance"] == 0).sum() / len(over1) * 100
print(f"\n% of UI>1 panchayats with opening_balance == 0: {ob_zero_pct:.1f}%")
# vs normal
ob_zero_norm = (normal["opening_balance"] == 0).sum() / len(normal) * 100
print(f"% of 0<UI<=1 panchayats with opening_balance == 0: {ob_zero_norm:.1f}%")

# =====================================================================
# PART B: Distribution of UI > 1 values — how extreme are they?
# =====================================================================
print(f"\n{'='*70}")
print("PART B: Distribution of UI > 1 values")
print(f"{'='*70}\n")

bins = [(1, 1.5), (1.5, 2), (2, 5), (5, 10), (10, 50), (50, float("inf"))]
print(f"{'UI Range':<15s} {'Count':>7s} {'% of >1':>9s} {'% of all':>9s}")
print("-" * 42)
for lo, hi in bins:
    label = f"{lo}-{hi}" if hi != float("inf") else f"{lo}+"
    count = ((over1["utilization_index"] > lo) & (over1["utilization_index"] <= hi)).sum()
    # include exact lower bound for first bin
    if lo == 1:
        count = (over1["utilization_index"] <= hi).sum()
    print(f"{label:<15s} {count:>7d} {count/len(over1)*100:>8.1f}% {count/len(df)*100:>8.1f}%")

print(f"\nSummary stats for UI > 1 group:")
print(over1["utilization_index"].describe().round(4))

# =====================================================================
# PART C: District-level distribution of UI > 1
# =====================================================================
print(f"\n{'='*70}")
print("PART C: Full district breakdown — UI > 1 counts and severity")
print(f"{'='*70}\n")

dist_total = df.groupby("district_name").size().reset_index(name="total_panchayats")

dist_over1 = over1.groupby("district_name").agg(
    n_over1=("utilization_index", "size"),
    median_ui=("utilization_index", "median"),
    max_ui=("utilization_index", "max"),
    median_excess=("excess", "median"),
).reset_index()

dist_over1 = dist_over1.merge(dist_total, on="district_name")
dist_over1["pct_over1"] = (dist_over1["n_over1"] / dist_over1["total_panchayats"] * 100).round(1)

# Sort by % affected
dist_over1 = dist_over1.sort_values("pct_over1", ascending=False)

print(dist_over1.round(2).to_string(index=False))

print(f"\n\nDistrict-level summary:")
print(f"  Districts with any UI>1: {len(dist_over1)}")
print(f"  Districts with >20% affected: {(dist_over1['pct_over1'] > 20).sum()}")
print(f"  Districts with >10% affected: {(dist_over1['pct_over1'] > 10).sum()}")
print(f"  Median % affected across districts: {dist_over1['pct_over1'].median():.1f}%")

Rows with UI > 1: 0
Rows with 0 < UI <= 1: 0

Column                           Median (UI>1)   Median (0<UI<=1)    Ratio
---------------------------------------------------------------------------
opening_balance                            nan                nan      nanx
total_receipts                             nan                nan      nanx
total_payments                             nan                nan      nanx
total_payment_cancellation                 nan                nan      nanx
total_receipt_cancellation                 nan                nan      nanx

Derived                          Median (UI>1)   Median (0<UI<=1)    Ratio
---------------------------------------------------------------------------
net_payments                               nan                nan      nanx
available_funds                            nan                nan      nanx

Median excess spending (net_payments - available_funds): nan
Mean excess spending: nan

% of UI>1 panchayats with open

C:\Users\anhad\AppData\Local\Temp\ipykernel_2132\1781611580.py:46: RuntimeWarning: invalid value encountered in scalar divide
  ob_zero_pct = (over1["opening_balance"] == 0).sum() / len(over1) * 100
C:\Users\anhad\AppData\Local\Temp\ipykernel_2132\1781611580.py:49: RuntimeWarning: invalid value encountered in scalar divide
  ob_zero_norm = (normal["opening_balance"] == 0).sum() / len(normal) * 100
C:\Users\anhad\AppData\Local\Temp\ipykernel_2132\1781611580.py:68: RuntimeWarning: invalid value encountered in scalar divide
  print(f"{label:<15s} {count:>7d} {count/len(over1)*100:>8.1f}% {count/len(df)*100:>8.1f}%")


## Check 16: Extreme opening balances (top/bottom 1%)

In [23]:
# Check if extreme opening balances are concentrated in specific districts
ob = df["opening_balance"].dropna()
p1 = ob.quantile(0.01)
p99 = ob.quantile(0.99)
print(f"Opening balance: 1st percentile = {p1:,.2f}, 99th percentile = {p99:,.2f}")
print(f"Overall range: {ob.min():,.2f} to {ob.max():,.2f}\n")

# Bottom 1%
bottom = df[df["opening_balance"] <= p1]
print(f"Bottom 1% ({len(bottom)} panchayats) — district distribution:")
print(bottom["district_name"].value_counts().head(10).to_string())

print()

# Top 1%
top = df[df["opening_balance"] >= p99]
print(f"Top 1% ({len(top)} panchayats) — district distribution:")
print(top["district_name"].value_counts().head(10).to_string())

# Check if extremes are concentrated
for label, subset in [("bottom 1%", bottom), ("top 1%", top)]:
    top_dist_share = subset["district_name"].value_counts(normalize=True).iloc[0] * 100
    top_dist_name = subset["district_name"].value_counts().index[0]
    if top_dist_share > 30:
        print(f"\nWARNING: {top_dist_share:.0f}% of {label} is from {top_dist_name} alone")

Opening balance: 1st percentile = 12,367.66, 99th percentile = 6,584,100.32
Overall range: 0.00 to 977,542,093.70

Bottom 1% (587 panchayats) — district distribution:
district_name
Saharanpur       77
Sultanpur        54
Budaun           45
Azamgarh         32
Basti            31
Ballia           25
Muzaffarnagar    20
Ghazipur         19
Amroha           16
Bijnor           14

Top 1% (587 panchayats) — district distribution:
district_name
Deoria               74
Kasganj              42
Maharajganj          33
Baghpat              33
Hapur                24
Unnao                22
Bareilly             18
Sultanpur            17
Sant Kabeer Nagar    17
Sonbhadra            14


## Check 17: Panchayats per block distribution — flag unusually small/large blocks

In [24]:
# Blocks with very few or very many panchayats could indicate merge errors or missing data
block_sizes = df.groupby(["district_name", "block_name"]).size().reset_index(name="n_panchayats")

print(f"Total blocks: {len(block_sizes)}")
print(f"Panchayats per block — min: {block_sizes['n_panchayats'].min()}, "
      f"max: {block_sizes['n_panchayats'].max()}, "
      f"median: {block_sizes['n_panchayats'].median():.0f}, "
      f"mean: {block_sizes['n_panchayats'].mean():.1f}\n")

# Flag blocks with only 1-2 panchayats (suspicious)
tiny_blocks = block_sizes[block_sizes["n_panchayats"] <= 2]
if len(tiny_blocks) > 0:
    print(f"WARNING: {len(tiny_blocks)} block(s) have <= 2 panchayats (possible missing data):")
    print(tiny_blocks.sort_values("n_panchayats").to_string(index=False))
else:
    print("PASS: No blocks with <= 2 panchayats")

print()

# Flag blocks that are >3σ above mean
mean_bs = block_sizes["n_panchayats"].mean()
std_bs = block_sizes["n_panchayats"].std()
large_blocks = block_sizes[block_sizes["n_panchayats"] > mean_bs + 3 * std_bs]
if len(large_blocks) > 0:
    print(f"WARNING: {len(large_blocks)} block(s) have unusually many panchayats (>3σ):")
    print(large_blocks.sort_values("n_panchayats", ascending=False).to_string(index=False))
else:
    print("PASS: No abnormally large blocks")

Total blocks: 827
Panchayats per block — min: 1, max: 188, median: 70, mean: 70.9

district_name block_name  n_panchayats
    Gorakhpur  Bharohiya             1

 district_name block_name  n_panchayats
      Pilibhit   PURANPUR           188
        Bijnor    KOTWALI           149
        Rampur       SUAR           149
Ambedkar Nagar   AKBARPUR           136
        Amroha       JOYA           136


## Check 18: Identical financial data across different panchayats (copy-paste detection)

In [25]:
# Two different panchayats with identical financial data across ALL monthly columns
# is highly suspicious (copy-paste error in source data)
# Exclude all-zero rows since those are legitimately common

months = ["april", "may", "june", "july", "august", "september",
          "october", "november", "december", "january", "february", "march"]
fin_fields = ["receipts", "payments", "receipt_cancellation", "payment_cancellation", "ob_rejected"]
monthly_cols = [f"{m}_{f}" for m in months for f in fin_fields]

# Exclude rows where all monthly columns are zero
all_zero_mask = (df[monthly_cols] == 0).all(axis=1)
df_nonzero = df[~all_zero_mask]

# Find duplicated financial fingerprints
dupes = df_nonzero.duplicated(subset=monthly_cols, keep=False)
n_dupe_groups = df_nonzero[dupes].groupby(monthly_cols).ngroups if dupes.sum() > 0 else 0

print(f"Non-zero rows: {len(df_nonzero)}")
print(f"Rows with identical financial data as another panchayat: {dupes.sum()}")
print(f"Distinct duplicate groups: {n_dupe_groups}")

if dupes.sum() > 0:
    # Show a sample of duplicate groups
    dupe_rows = df_nonzero[dupes].sort_values(monthly_cols[:5])
    sample_group = dupe_rows.groupby(monthly_cols).first().head(3).index
    print("\nSample duplicate groups:")
    for i, key in enumerate(sample_group):
        match = df_nonzero[(df_nonzero[monthly_cols] == pd.Series(dict(zip(monthly_cols, key)))).all(axis=1)]
        print(f"\n  Group {i+1} ({len(match)} rows):")
        print(match[["district_name", "block_name", "panchayat_name", "panchayat_code"]].to_string(index=False))
else:
    print("PASS: No suspicious duplicate financial fingerprints")

Non-zero rows: 58664
Rows with identical financial data as another panchayat: 10
Distinct duplicate groups: 5

Sample duplicate groups:

  Group 1 (2 rows):
district_name block_name  panchayat_name  panchayat_code
       Etawah   BASREHAR SHERPURBASGAWAN           60522
       Etawah   BASREHAR SHERPURBASGAWAN           60522

  Group 2 (2 rows):
district_name block_name panchayat_name  panchayat_code
    Prayagraj       MEJA     CHAPARTALA          268353
    Prayagraj       MEJA    SALAIYAKALA          268457

  Group 3 (2 rows):
district_name block_name panchayat_name  panchayat_code
      Aligarh   CHANDAUS       MOHARANA           43591
      Aligarh   CHANDAUS       MOHRAINA           43591


## Check 19: Months with zero activity across entire district (data collection gaps)

In [26]:
# If ALL panchayats in a district have 0 receipts AND 0 payments for a given month,
# that's likely a data collection failure, not reality

months = ["april", "may", "june", "july", "august", "september",
          "october", "november", "december", "january", "february", "march"]

dead_months = []
for district in df["district_name"].unique():
    dist_df = df[df["district_name"] == district]
    for m in months:
        r_col = f"{m}_receipts"
        p_col = f"{m}_payments"
        if (dist_df[r_col] == 0).all() and (dist_df[p_col] == 0).all():
            dead_months.append((district, m, len(dist_df)))

if not dead_months:
    print("PASS: No district has a completely zero-activity month")
else:
    print(f"WARNING: {len(dead_months)} district-month combinations have zero activity across ALL panchayats:\n")
    dead_df = pd.DataFrame(dead_months, columns=["district", "month", "n_panchayats"])
    # Pivot to see pattern
    pivot = dead_df.pivot_table(index="district", columns="month", values="n_panchayats", aggfunc="size", fill_value=0)
    # Reorder months
    pivot = pivot.reindex(columns=[m for m in months if m in pivot.columns])
    print(pivot.to_string())
    
    # Count dead months per district
    per_dist = dead_df.groupby("district").size().sort_values(ascending=False)
    print(f"\nDead months per district:")
    print(per_dist.to_string())

PASS: No district has a completely zero-activity month


## Check 20: District-level distribution (all 4 metrics)

In [27]:
# Per-district summary for all 4 metrics
# Flag: median near 0 (< 0.1), median >= 1, high CV (> 2)
metrics = ["utilization_index", "utilization_index_no_canc", "payment_ratio", "payment_ratio_no_canc"]

for metric in metrics:
    print(f"{'='*70}")
    print(f"District-level summary: {metric}")
    print(f"{'='*70}")
    
    dist_m = df.groupby("district_name")[metric].agg(
        ["median", "mean", "std", "count",
         lambda x: x.isna().sum()]
    ).rename(columns={"<lambda_0>": "n_nan"})
    dist_m["cv"] = (dist_m["std"] / dist_m["mean"]).round(2)
    dist_m = dist_m.round(4)
    
    # Flag: median near 0
    low_median = dist_m[dist_m["median"] < 0.1]
    print(f"\nDistricts with median < 0.1:")
    if len(low_median) > 0:
        print(low_median.sort_values("median")[["median", "mean", "std", "n_nan"]].to_string())
    else:
        print("  None")
    
    # Flag: median >= 1
    high_median = dist_m[dist_m["median"] >= 1]
    print(f"\nDistricts with median >= 1:")
    if len(high_median) > 0:
        print(high_median.sort_values("median", ascending=False)[["median", "mean", "std", "n_nan"]].to_string())
    else:
        print("  None")
    
    # Flag: high within-district variation
    high_cv = dist_m[dist_m["cv"] > 2].sort_values("cv", ascending=False)
    print(f"\nDistricts with CV > 2 (high variation):")
    if len(high_cv) > 0:
        print(high_cv[["median", "mean", "std", "cv"]].to_string())
    else:
        print("  None")
    
    # Overall summary
    print(f"\nOverall district-level stats:")
    print(dist_m[["median", "mean", "std", "cv"]].describe().round(4))
    print()

District-level summary: utilization_index

Districts with median < 0.1:
                     median  mean  std  n_nan
district_name                                
Agra                    0.0   0.0  0.0      0
Meerut                  0.0   0.0  0.0      0
Mau                     0.0   0.0  0.0      0
Mathura                 0.0   0.0  0.0      0
Mainpuri                0.0   0.0  0.0      0
Mahoba                  0.0   0.0  0.0      0
Maharajganj             0.0   0.0  0.0      0
Lucknow                 0.0   0.0  0.0      0
Lalitpur                0.0   0.0  0.0      0
Kushi Nagar             0.0   0.0  0.0      0
Kheri                   0.0   0.0  0.0      2
Kaushambi               0.0   0.0  0.0      0
Kasganj                 0.0   0.0  0.0      0
Kanpur Nagar            0.0   0.0  0.0      0
Kanpur Dehat            0.0   0.0  0.0      0
Kannauj                 0.0   0.0  0.0      0
Mirzapur                0.0   0.0  0.0      0
Jhansi                  0.0   0.0  0.0      0
Moradaba

### Check 20b: Diagnose negative values (all 4 metrics)

In [28]:
# Negative metric values: diagnose root cause for each of the 4 metrics
# For "with cancellations" variants: negative numerator = cancellations > gross amount
# For "no_canc" variants: numerator is just total_payments (always >= 0),
#   so negatives should be impossible — any found = data corruption

metrics_info = {
    "utilization_index": {
        "num": ("total_payments", "total_payment_cancellation"),  # payments - pay_canc
        "can_be_negative": True,
    },
    "utilization_index_no_canc": {
        "num": ("total_payments",),  # just payments
        "can_be_negative": False,
    },
    "payment_ratio": {
        "num": ("total_payments", "total_payment_cancellation"),
        "can_be_negative": True,
    },
    "payment_ratio_no_canc": {
        "num": ("total_payments",),
        "can_be_negative": False,
    },
}

for metric, info in metrics_info.items():
    neg_m = df[df[metric] < 0].copy()
    print(f"{'='*70}")
    print(f"{metric}: {len(neg_m)} negative values\n")
    
    if len(neg_m) == 0:
        print("PASS: No negative values\n")
        continue
    
    if not info["can_be_negative"]:
        print("ERROR: This metric should NEVER be negative (no cancellation in formula).")
        print("These rows likely have data corruption.\n")
    
    # Magnitude breakdown
    tiny = (neg_m[metric] >= -0.01).sum()
    substantial = (neg_m[metric] < -0.01).sum()
    print(f"  Near-zero negatives (>= -0.01, rounding): {tiny}")
    print(f"  Substantial negatives (< -0.01):          {substantial}")
    print(f"  Range: {neg_m[metric].min():.4f} to {neg_m[metric].max():.4f}")
    
    # District breakdown (substantial only)
    large_neg = neg_m[neg_m[metric] < -0.01]
    if len(large_neg) > 0:
        print(f"\n  Substantial negatives by district:")
        by_dist = large_neg.groupby("district_name").agg(
            n=("district_name", "size"),
            median_val=(metric, "median"),
            min_val=(metric, "min"),
        ).sort_values("n", ascending=False)
        print(by_dist.head(10).round(4).to_string())
    
    # Worst rows
    print(f"\n  5 most negative rows:")
    worst = neg_m.nsmallest(5, metric)[
        ["district_name", "panchayat_name",
         "total_payments", "total_payment_cancellation",
         "opening_balance", "total_receipts", "total_receipt_cancellation",
         metric]
    ]
    print(worst.to_string(index=False))
    print()

# Month-level check: cancellations > payments
months = ["april","may","june","july","august","september",
          "october","november","december","january","february","march"]
print(f"{'='*70}")
print("Monthly aggregate: months where payment_cancellations > payments")
found_any = False
for m in months:
    p  = df[f"{m}_payments"].sum()
    pc = df[f"{m}_payment_cancellation"].sum()
    if pc > p:
        found_any = True
        print(f"  {m:>10s}: payments={p:,.0f}  cancellations={pc:,.0f}  EXCESS={pc-p:,.0f}")
if not found_any:
    print("  None — all months have payments >= cancellations in aggregate")

utilization_index: 0 negative values

PASS: No negative values

utilization_index_no_canc: 0 negative values

PASS: No negative values

payment_ratio: 0 negative values

PASS: No negative values

payment_ratio_no_canc: 0 negative values

PASS: No negative values

Monthly aggregate: months where payment_cancellations > payments
  None — all months have payments >= cancellations in aggregate


## Check 21: Row count vs state average — flag districts with unusually few panchayats

In [29]:
# Districts with significantly fewer panchayats than average could indicate
# truncated file exports or missing blocks

dist_sizes = df.groupby("district_name").size().reset_index(name="n_panchayats")
mean_size = dist_sizes["n_panchayats"].mean()
std_size = dist_sizes["n_panchayats"].std()

# Flag districts more than 2σ below mean
low_threshold = mean_size - 2 * std_size
small_districts = dist_sizes[dist_sizes["n_panchayats"] < low_threshold].sort_values("n_panchayats")

print(f"Mean panchayats per district: {mean_size:.0f}")
print(f"Std: {std_size:.0f}")
print(f"Flag threshold (mean - 2σ): {low_threshold:.0f}\n")

if len(small_districts) > 0:
    print(f"WARNING: {len(small_districts)} district(s) have unusually few panchayats (<2σ below mean):")
    print(small_districts.to_string(index=False))
else:
    print("PASS: No districts with unusually few panchayats")

# Also show the full distribution for context
print(f"\nFull district size distribution:")
print(dist_sizes["n_panchayats"].describe().to_string())

Mean panchayats per district: 782
Std: 375
Flag threshold (mean - 2σ): 31

PASS: No districts with unusually few panchayats

Full district size distribution:
count      75.000000
mean      782.186667
std       375.409639
min       100.000000
25%       500.000000
50%       695.000000
75%      1046.000000
max      1869.000000


## Check 22: All-zero panchayats (completely inactive)

In [30]:
# Panchayats where every monthly receipt AND payment is 0
# Could be ghost entries, data collection failures, or genuinely new/dormant panchayats

months = ["april", "may", "june", "july", "august", "september",
          "october", "november", "december", "january", "february", "march"]
receipt_cols = [f"{m}_receipts" for m in months]
payment_cols = [f"{m}_payments" for m in months]

all_zero = (df[receipt_cols + payment_cols] == 0).all(axis=1)
zero_rows = df[all_zero]

print(f"Completely inactive panchayats (all monthly receipts & payments = 0): {len(zero_rows)} / {len(df)}")

if len(zero_rows) > 0:
    # Do they still have opening balance?
    has_ob = (zero_rows["opening_balance"] > 0).sum()
    print(f"  Of these, {has_ob} have a non-zero opening balance (dormant accounts with funds)")
    
    # District breakdown
    zero_by_dist = (
        zero_rows.groupby("district_name").size()
        .reset_index(name="zero_count")
        .sort_values("zero_count", ascending=False)
    )
    dist_total = df.groupby("district_name").size().reset_index(name="total")
    zero_by_dist = zero_by_dist.merge(dist_total, on="district_name")
    zero_by_dist["pct"] = (zero_by_dist["zero_count"] / zero_by_dist["total"] * 100).round(1)
    
    print(f"\nDistricts with most all-zero panchayats:")
    print(zero_by_dist.head(15).to_string(index=False))

Completely inactive panchayats (all monthly receipts & payments = 0): 32 / 58664
  Of these, 32 have a non-zero opening balance (dormant accounts with funds)

Districts with most all-zero panchayats:
  district_name  zero_count  total  pct
      Gorakhpur           7   1343  0.5
    Farrukhabad           5    600  0.8
       Mainpuri           4    549  0.7
          Gonda           3   1053  0.3
       Azamgarh           2   1869  0.1
       Bahraich           2   1054  0.2
       Ghazipur           2   1235  0.2
           Agra           1    695  0.1
        Aligarh           1    888  0.1
      Barabanki           1   1166  0.1
         Deoria           1   1184  0.1
            Mau           1    675  0.1
      Prayagraj           1   1636  0.1
Siddharth Nagar           1   1199  0.1


## Check 23: Closing balance reconstruction — flag negative closing balances

In [31]:
# closing = opening_balance + total_receipts - total_receipt_cancellation
#           - total_payments + total_payment_cancellation - total_ob_rejected
# Negative closing balance shouldn't normally happen

df["closing_balance"] = (
    df["opening_balance"]
    + df["total_receipts"] - df["total_receipt_cancellation"]
    - df["total_payments"] + df["total_payment_cancellation"]
    - df["total_ob_rejected"]
)

neg_closing = df[df["closing_balance"] < 0]
print(f"Panchayats with negative closing balance: {len(neg_closing)} / {len(df)}")

if len(neg_closing) > 0:
    print(f"\nClosing balance range for negative rows: {neg_closing['closing_balance'].min():,.2f} to {neg_closing['closing_balance'].max():,.2f}")
    
    # District breakdown
    neg_by_dist = (
        neg_closing.groupby("district_name").size()
        .reset_index(name="neg_count")
        .sort_values("neg_count", ascending=False)
    )
    print(f"\nNegative closing balances by district (top 10):")
    print(neg_by_dist.head(10).to_string(index=False))
    
    # Show the worst offenders
    print(f"\nTop 10 most negative closing balances:")
    worst = neg_closing.nsmallest(10, "closing_balance")[
        ["district_name", "panchayat_name", "opening_balance",
         "total_receipts", "total_payments", "total_ob_rejected", "closing_balance"]
    ]
    print(worst.to_string(index=False))
else:
    print("PASS: No negative closing balances")

Panchayats with negative closing balance: 0 / 58664
PASS: No negative closing balances


## Check 24: Cancellation ratios — flag high cancellation rates

In [32]:
# High cancellation rates (>20%) at the panchayat level could indicate
# accounting corrections or data entry errors

# Receipt cancellation ratio
has_receipts = df["total_receipts"] > 0
df.loc[has_receipts, "receipt_canc_ratio"] = (
    df.loc[has_receipts, "total_receipt_cancellation"] / df.loc[has_receipts, "total_receipts"]
)

# Payment cancellation ratio
has_payments = df["total_payments"] > 0
df.loc[has_payments, "payment_canc_ratio"] = (
    df.loc[has_payments, "total_payment_cancellation"] / df.loc[has_payments, "total_payments"]
)

for label, col in [("Receipt", "receipt_canc_ratio"), ("Payment", "payment_canc_ratio")]:
    valid = df[col].dropna()
    high = df[df[col] > 0.2]
    print(f"{label} cancellation ratio:")
    print(f"  Valid rows: {len(valid)}")
    print(f"  Mean: {valid.mean():.4f}, Median: {valid.median():.4f}, Max: {valid.max():.4f}")
    print(f"  Panchayats with >20% cancellation: {len(high)}")
    
    if len(high) > 0:
        high_by_dist = high.groupby("district_name").size().sort_values(ascending=False)
        print(f"  Top districts with high {label.lower()} cancellations:")
        print(high_by_dist.head(10).to_string())
    print()

Receipt cancellation ratio:
  Valid rows: 58632
  Mean: 2.4744, Median: 1.0176, Max: 84583.3333
  Panchayats with >20% cancellation: 58324
  Top districts with high receipt cancellations:
district_name
Azamgarh           1861
Jaunpur            1736
Prayagraj          1626
Sitapur            1568
Gorakhpur          1325
Hardoi             1303
Pratapgarh         1235
Ghazipur           1223
Siddharth Nagar    1192
Bareilly           1191

Payment cancellation ratio:
  Valid rows: 0
  Mean: nan, Median: nan, Max: nan
  Panchayats with >20% cancellation: 0



## Check 25: Opening balance vs activity consistency (dormant accounts)

In [33]:
# Panchayats with a large opening balance but zero total_receipts AND zero total_payments
# These are dormant accounts — funds sitting idle the entire year

dormant = df[
    (df["opening_balance"] > 0) &
    (df["total_receipts"] == 0) &
    (df["total_payments"] == 0)
]

print(f"Dormant panchayats (OB > 0 but zero receipts & payments all year): {len(dormant)} / {len(df)}")

if len(dormant) > 0:
    print(f"Total idle funds in dormant accounts: ₹{dormant['opening_balance'].sum():,.2f}")
    print(f"Opening balance range: ₹{dormant['opening_balance'].min():,.2f} to ₹{dormant['opening_balance'].max():,.2f}")
    
    dormant_by_dist = (
        dormant.groupby("district_name")
        .agg(n_dormant=("panchayat_name", "size"),
             idle_funds=("opening_balance", "sum"))
        .sort_values("n_dormant", ascending=False)
    )
    print(f"\nDormant panchayats by district (top 10):")
    dormant_by_dist["idle_funds"] = dormant_by_dist["idle_funds"].apply(lambda x: f"₹{x:,.0f}")
    print(dormant_by_dist.head(10).to_string())
else:
    print("PASS: No dormant accounts found")

Dormant panchayats (OB > 0 but zero receipts & payments all year): 32 / 58664
Total idle funds in dormant accounts: ₹53,864,234.82
Opening balance range: ₹228,614.00 to ₹4,270,534.56

Dormant panchayats by district (top 10):
               n_dormant   idle_funds
district_name                        
Gorakhpur              7  ₹15,686,474
Farrukhabad            5   ₹9,349,130
Mainpuri               4   ₹8,125,949
Gonda                  3   ₹2,760,037
Azamgarh               2   ₹1,242,981
Bahraich               2   ₹4,726,874
Ghazipur               2   ₹4,322,700
Agra                   1     ₹418,995
Aligarh                1   ₹2,400,486
Barabanki              1   ₹1,211,714


## Check 26: March rush spending pattern

In [34]:
# March is typically the "rush spending" month in Indian govt accounting
# Quantify how much of annual payments happen in March

months = ["april", "may", "june", "july", "august", "september",
          "october", "november", "december", "january", "february", "march"]

monthly_totals = {}
for m in months:
    monthly_totals[m] = df[f"{m}_payments"].sum()

monthly_series = pd.Series(monthly_totals)
total_payments = monthly_series.sum()

print("Monthly share of total payments:")
for m in months:
    share = monthly_totals[m] / total_payments * 100
    bar = "█" * int(share)
    print(f"  {m:>10s}: {share:5.1f}%  {bar}")

march_share = monthly_totals["march"] / total_payments * 100
print(f"\nMarch accounts for {march_share:.1f}% of all annual payments")
if march_share > 25:
    print("WARNING: March rush spending is very pronounced (>25%)")

# Per-district March share
df["march_payment_share"] = np.where(
    df["total_payments"] > 0,
    df["march_payments"] / df["total_payments"],
    np.nan
)
dist_march = df.groupby("district_name")["march_payment_share"].median().sort_values(ascending=False)
print(f"\nDistricts with highest median March payment share:")
print(dist_march.head(10).apply(lambda x: f"{x:.1%}").to_string())

Monthly share of total payments:


C:\Users\anhad\AppData\Local\Temp\ipykernel_2132\3696420815.py:16: RuntimeWarning: invalid value encountered in scalar divide
  share = monthly_totals[m] / total_payments * 100


ValueError: cannot convert float NaN to integer

## Export: Save cleaned DataFrame to CSV

In [ ]:
output_file = "combined_data_2018-19_cleaned.csv"
df.to_csv(output_file, index=False)
print(f"Saved cleaned DataFrame to '{output_file}'")
print(f"  Rows: {len(df)}")
print(f"  Columns: {len(df.columns)}")
print(f"  Columns: {list(df.columns)}")

## Check 27: Prevalence of zeros in total financial columns

In [ ]:
# How many panchayats have exactly 0 in each total financial column?
total_cols = ["opening_balance", "total_receipts", "total_payments",
              "total_receipt_cancellation", "total_payment_cancellation", "total_ob_rejected"]

n = len(df)
print(f"Total panchayats: {n}\n")
print(f"{'Column':<30s} {'Zeros':>7s} {'% Zero':>8s} {'NaN':>6s}")
print("-" * 55)
for col in total_cols:
    n_zero = (df[col] == 0).sum()
    n_nan = df[col].isna().sum()
    print(f"{col:<30s} {n_zero:>7d} {n_zero/n*100:>7.1f}% {n_nan:>6d}")

# District-level breakdown for key columns (receipts, payments, OB)
print("\n\n--- District breakdown: panchayats with zero total_receipts ---")
zero_rec = df[df["total_receipts"] == 0]
zr_by_dist = zero_rec.groupby("district_name").size().reset_index(name="zero_receipts")
dist_total = df.groupby("district_name").size().reset_index(name="total")
zr_by_dist = zr_by_dist.merge(dist_total, on="district_name")
zr_by_dist["pct"] = (zr_by_dist["zero_receipts"] / zr_by_dist["total"] * 100).round(1)
zr_by_dist = zr_by_dist.sort_values("pct", ascending=False)
print(zr_by_dist.head(15).to_string(index=False))

print("\n\n--- District breakdown: panchayats with zero total_payments ---")
zero_pay = df[df["total_payments"] == 0]
zp_by_dist = zero_pay.groupby("district_name").size().reset_index(name="zero_payments")
zp_by_dist = zp_by_dist.merge(dist_total, on="district_name")
zp_by_dist["pct"] = (zp_by_dist["zero_payments"] / zp_by_dist["total"] * 100).round(1)
zp_by_dist = zp_by_dist.sort_values("pct", ascending=False)
print(zp_by_dist.head(15).to_string(index=False))

print("\n\n--- District breakdown: panchayats with zero opening_balance ---")
zero_ob = df[df["opening_balance"] == 0]
zo_by_dist = zero_ob.groupby("district_name").size().reset_index(name="zero_ob")
zo_by_dist = zo_by_dist.merge(dist_total, on="district_name")
zo_by_dist["pct"] = (zo_by_dist["zero_ob"] / zo_by_dist["total"] * 100).round(1)
zo_by_dist = zo_by_dist.sort_values("pct", ascending=False)
print(zo_by_dist.head(15).to_string(index=False))

# Combo: panchayats with ALL three key columns == 0 (completely empty rows)
all_three_zero = df[(df["opening_balance"] == 0) & (df["total_receipts"] == 0) & (df["total_payments"] == 0)]
print(f"\n\nPanchayats with OB=0 AND receipts=0 AND payments=0: {len(all_three_zero)} / {n} ({len(all_three_zero)/n*100:.1f}%)")
if len(all_three_zero) > 0:
    az_by_dist = all_three_zero.groupby("district_name").size().reset_index(name="all_zero")
    az_by_dist = az_by_dist.merge(dist_total, on="district_name")
    az_by_dist["pct"] = (az_by_dist["all_zero"] / az_by_dist["total"] * 100).round(1)
    print(az_by_dist.sort_values("pct", ascending=False).head(10).to_string(index=False))